In [1]:

from ait.photonic_testing.photonic_testing import *
MODBUS_PORT='/dev/tty.usbserial-B003T6PZ'
# MODBUS_PORT='/dev/tty.usbserial-B003T6RF'
LASER_ADDRESSES = {"1028": 5,
                   "1270": 6,
                   "yj1430": 4,
                   "hk1430": 3,
                   "1510": 1,
                   "2330": 2}

LASER_DRIVER_SERIALS = {"1028": 8229,
                   "1270": 8228,
                   "yj1430": 8222,
                   "hk1430": 8227,
                   "1510": 8225,
                   "2330": 8226}

### Overview

See `ait/photonic_testing.py` for `Laser` and `LaserProperties` along with the specific limits of the individual laser diodes.


### Low-level access

In the event that direct device control is needed.

See `ait/maiman_modbus/utils/utils.py` for python constants of register names and `ait/maiman_modbus/config/modbus_config.yaml` for the register addresses.

Look at methods on `ModbusDevice` (`ait/maiman_modbus/device/modbus_device.py`) for functions.


```python
from maiman_modbus.communication import ModbusCommunication
import maiman_modbus.utils as maiman_regs
from maiman_modbus.device.modbus_device_model import ModbusDeviceModel
from maiman_modbus.config import DeviceConfig
from maiman_modbus.device.modbus_device import ModbusDevice

d = ModbusDevice(port=MODBUS_PORT, slave_address=modbus_address)
d.comm.send_command(d.model.get_register(STATE_OF_TEC_COMMAND), MODBUS_START_TEC_COMMAND_VALUE)

print(d.comm.receive_response(d.model.get_register(STATE_OF_TEC_COMMAND)))
```


## Initialize all the diodes

Running this cell will create the `lasers` dictionary with a `Laser` for each laser.

In [2]:
names = tuple(LASER_ADDRESSES.keys())
lasers = {}
for name in names:
    l = Laser(name, address=LASER_ADDRESSES[name], MODBUS_PORT=MODBUS_PORT)
    serial = l.device.get_serial_number()
    print(f'🆔 Serial number: {serial}')
    assert l.device.get_serial_number()==LASER_DRIVER_SERIALS[name], 'BAD BUS CONFIG, do not continue'
    lasers[name] = l
    print('')


for name in names:
    serial = lasers[name].device.get_serial_number()
    assert serial==LASER_DRIVER_SERIALS[name], 'BAD BUS CONFIG, do not continue'



Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.
🆔 Serial number: 8229

Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.
🆔 Serial number: 8228

Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.
🆔 Serial number: 8222

Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.
🆔 Serial number: 8227

Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.
🆔 Serial number: 8225

Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.
🆔 Serial number: 8226



### Turn on a laser

This sets a percentage between the maximum current and the threshold current.

In [4]:
l= lasers['1270']
l.disable_interlock_and_cool()
l.set_current_as_percent(.2)
print(l.laser_properties.max_current, l.device.get_current_protection_threshold())

Programming limits for 1270, Maiman driver: S/N 8228...
tec_running: True, interlocked: False, started: True
Setting current to 18.4 mA 
...current: 18.4 mA


18.4

Look at the status of one of them. It seems that the TEC status isn't polling well, but the temp changes.

In [ ]:
l.status()

In [9]:
l.shutdown()
l.status()

Laser Properties Name: 1028
Raw ID from register: 0x1113
Device ID: 4371
Serial Number: 8229
State: 0x75
 Operation started: False
 Current Set Internal: True
 Enable Internal: True
 External NTC Denied: True
 Interlock Denied: False
Current: 0.0
Current Min: 0.0
Current Max: 225.0
Protection Threshold: 248.9
Driver Max Current: 250.0
Voltage: 0.0
Frequency: 0.0
Duration: 1.0
Raw PCB temperature (signed): 0
PCB Temp: 0.0
Current Set Calibration: 100.0
TEC PID: (20, 1000, 1000)
TEC Voltage: 0.0
TEC Current Limit: 1.2
TEC Current: 0.0
TEC Temperature Setpoint: 25.0
TEC Temperature: 25.21
TEC NTC Coefficient: 3988.0
TEC State: 0x14
 TEC started: False
 TEC Set Internal: True
 TEC Enable Internal: True
Interlock State: 0x2
 Interlock: True
 LD Overcurrent: False
 LD Overheat: False
 External NTC Interlock: False
 TEC Error: False
 TEC Self-heat: False


In [5]:
for k, v in lasers.items():
    v.set_current_as_percent(.67)

Programming limits for 1028, Maiman driver: S/N 8229...
Setting current to 155.535 mA 
...current: 155.5 mA
Programming limits for 1270, Maiman driver: S/N 8228...
Setting current to 42.84 mA 
...current: 42.8 mA
Programming limits for yj1430, Maiman driver: S/N 8222...
Setting current to 42.84 mA 
...current: 42.8 mA
Programming limits for hk1430, Maiman driver: S/N 8227...
Setting current to 42.84 mA 
...current: 42.8 mA
Programming limits for 1510, Maiman driver: S/N 8225...
Setting current to 42.84 mA 
...current: 42.8 mA
Programming limits for 2330, Maiman driver: S/N 8226...
Setting current to 81.917 mA 
...current: 81.9 mA


In [ ]:
# This cell's code looks at the low lever current values in the respective registers, I've emailed Maiman about unexpected tripping of the OCP. -JIB 7/16/25
# import maiman_modbus.utils as maiman_regs
# for k in lasers:
#     laser=lasers[k]
#     self=laser.device
#     raw_ocp = self.comm.receive_response(self.model.get_register(maiman_regs.REGISTER_CURRENT_PROTECTION_THRESHOLD))
#
#     max_current = laser.laser_properties.max_current.to(u.mA).value-OC_POT_TEMP_DRIFT_MARGIN_MA
#
#     print(f'{k}: {self.get_current_protection_threshold()} ({raw_ocp}), {self.get_current_max()} ({int(self.get_divider(maiman_regs.REGISTER_CURRENT)*max_current)})')
#
# for k, v in lasers.items(): print(k, v.device.get_current_max(), v.device.get_current())

### Make sure it is all off.

In [5]:
for name in names:
    lasers[name].shutdown()

0.14A idle with TEC

1.46A with all @ 75%

1.56A with all @ 90%

1.59 with all at 95%
